[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/rolling_ball/rolling_ball.ipynb)

# A ball rolled around a loop

A ball that rolls without slipping on a table turns in the plane of the vertical and its motion. Moving by a small step $d$, a ball of radius $r$ turns by the rotor $e^{-(z \wedge d) / 2r}$: the outer product of the vertical with the step, the plane it rolls in, scaled by the angle the step rolls it through. The ball's orientation is the product of these rotors along the path. Rolled once around a closed loop, the steps do not undo each other, and the ball comes back turned: for a small loop, about the vertical, by the area of the loop over the square of the radius, the outer products of the steps summed. The point where the ball touches the table draws a trail on the ball, a curve on its surface.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np
from IPython.display import Image, display

from numga import NumpyContext
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.mechanics.rolling_ball import render

ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
RADIUS = 1.0
# The loop: a circle of this radius, rolled around this many times.
LOOP = 2.5
LOOPS = 3
STEPS_PER_LOOP = 720
FRAMES_PER_LOOP = 45
RESOLUTION = 200
FRAME_MS = 50

## 1. Rolling along the loop

The ball's centre runs around the circle, the circle's points turned from $x$ by the rotor in the $xy$ plane. Each step turns the ball by the rotor of the plane of $z$ and the step; the running product of these rotors, each later one multiplying from the left, is the orientation that takes the ball's own frame to the table's. The point of the ball touching the table, in the ball's own frame, is the downward direction taken back through the orientation.

In [ ]:
angles = np.linspace(0.0, 2 * np.pi * LOOPS, LOOPS * STEPS_PER_LOOP + 1)     # [steps + 1]
path = ((mv.x ^ mv.y) * (-angles / 2)).exp() >> (LOOP * mv.x)                  # [steps + 1] Vector
steps = path[1:] - path[:-1]                                                    # [steps] Vector
turns = ((mv.z ^ steps) * (-1 / (2 * RADIUS))).exp()                            # [steps] Rotor
orientation = turns.cumprod(axis=0)                                             # [steps] Rotor
contact = orientation << (-RADIUS * mv.z)                                       # [steps] Vector

## 2. Once around

After one loop the ball has turned by the product of the loop's rotors. The loop's area is half the sum of the outer products of successive points: a bivector in the plane of the table.

In [ ]:
once = orientation[STEPS_PER_LOOP - 1]                                          # [] Rotor
area = 0.5 * (path[:STEPS_PER_LOOP] ^ path[1:STEPS_PER_LOOP + 1]).sum()         # [] Bivector
print("the turn after one loop:")
print(once.coefficient_table())
print("the area of the loop:")
print(area.coefficient_table())

## 3. The ball on the table

Seen from above, the ball rolls around the loop. Each pixel of it shows the point of the ball beneath, taken back into the ball's own frame, where the pattern and the trail are fixed.

In [ ]:
def rolling(frames_per_loop: int):
    """Per frame, the ball's position, the point of the ball under each pixel in its own frame, and
    the trail its contact point has drawn so far."""
    for step in np.linspace(0, len(orientation) - 1, frames_per_loop * LOOPS).astype(int):
        yield path[step + 1], orientation[step] << screen, contact[:step + 1]


u, v = render.disk(RESOLUTION)
screen = mv.x * u + mv.y * v + mv.z * np.sqrt(np.clip(1 - u**2 - v**2, 0.0, None))   # [rows, columns] Vector
display(Image(filename=save_animation(render.animate(path, screen, rolling(FRAMES_PER_LOOP), RADIUS), "rolling_ball", FRAME_MS)))

In [ ]:
# checks
# The rotors keep the ball rigid; the contact point stays on the ball; for a loop small against the
# ball, the turn after one loop is the exponential of half the loop's area over the square of the
# radius, about the vertical.
np.testing.assert_allclose(orientation.symmetric_reverse_product().to_array(), 1.0, atol=1e-9)
np.testing.assert_allclose(contact.squared().to_array(), RADIUS**2, atol=1e-9)
small = ((mv.x ^ mv.y) * (-angles[:STEPS_PER_LOOP + 1] / 2)).exp() >> (0.05 * RADIUS * mv.x)   # [steps + 1] Vector
small_once = ((mv.z ^ (small[1:] - small[:-1])) * (-1 / (2 * RADIUS))).exp().cumprod(axis=0)[-1]
small_area = 0.5 * (small[:-1] ^ small[1:]).sum()                               # [] Bivector
predicted = (small_area / (2 * RADIUS**2)).exp()                                # [] Rotor
np.testing.assert_allclose((small_once - predicted).kernel, 0.0, atol=3e-4)